## Opay Bank Statement Parser

In [4]:
import pymupdf
import pandas as pd
import re

## Extracting and Loading Data

In [5]:
tables = []
with pymupdf.open('opay_bankstatement.pdf') as doc:
    print(len(doc))
    for page_number in range(len(doc)):
        page = doc.load_page(page_number)
        text_blocks = page.get_text('blocks')

        sorted_blocks = sorted(text_blocks, key=lambda b: b[1])

        table_data = []
        for block in sorted_blocks:
            # print(block)
            lines = block[4].split('\n')
            # print(lines)
            table_data.append(lines)

        if table_data:
            df = pd.DataFrame(table_data)
            processed_df = df
            # print(processed_df)

            if not processed_df.empty:
                # print(processed_df)
                tables.append(processed_df)

    if tables:
        concatenated_df = pd.concat(tables, ignore_index=True)
        bank_statement = concatenated_df
    else:
        bank_statement = pd.DataFrame()

33


In [6]:
# Renaming Columns
col = ['Trans.Time', 'Value Date', 'Description', 'Debit/Credit(#)', 'Balance(#)', 'Channel', 'Transaction Reference',
'NoneDrop']
bank_statement.columns = col

In [7]:
# Dropping multiple headers and rows with irrelevant values.
bs_df = bank_statement.copy()
bs_df = bs_df.drop(bs_df[bs_df['Trans.Time']=='Trans. Time'].index)
bs_df = bs_df[~bs_df['Trans.Time'].str.contains('^[A-Z]', regex=True)]
bs_df = bs_df[~bs_df['Trans.Time'].str.contains('^₦', regex=True)]

In [ ]:
date_pattern = re.compile(r"\d{2} [A-Za-z]{3} \d{4}$")
dt_complete_mapper = re.compile(r'^\d{4}\s(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s\d{2}\s\d{2}:\d{2}:\d{2}$')
# dt_incomplete_mapper = re.compile(r'^\d{4}\s(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s\d{2}\s\d{2}:\d{2}:$')

bs_df = bs_df.copy()
def check_datetime(row):
    cell_1 = str(row[0])
    return bool(
        re.match(date_pattern, cell_1)
        or
        re.match(dt_complete_mapper, cell_1))
        # or 
        # re.match(dt_incomplete_mapper, cell_1))

bs_df = bs_df[bs_df.apply(check_datetime, axis=1)]
bs_df.to_csv('open_second.csv')


#### Tasks
- Shift rows with cell in column(Trans.Time) that matches [dd Month Year] format to the right.
- Delete Trans.Time column.
- Drop None values

#### Separating columns dataframe based on shifting goal

In [9]:
nas_dropped = bs_df.dropna(subset=['Balance(#)']).reset_index()
df_to_shift = nas_dropped[~(nas_dropped['Channel'] == 'E-Channel')].reset_index()
df_stable = nas_dropped[nas_dropped['Channel'] == 'E-Channel'].reset_index()

df_stable.info()
df_to_shift.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 202 entries, 0 to 201
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   level_0                202 non-null    int64 
 1   index                  202 non-null    int64 
 2   Trans.Time             202 non-null    object
 3   Value Date             202 non-null    object
 4   Description            202 non-null    object
 5   Debit/Credit(#)        202 non-null    object
 6   Balance(#)             202 non-null    object
 7   Channel                202 non-null    object
 8   Transaction Reference  202 non-null    object
 9   NoneDrop               201 non-null    object
dtypes: int64(2), object(8)
memory usage: 15.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 542 entries, 0 to 541
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   level_0             

### Performing shifting on dataframe selected.

In [10]:
shift_criteria = df_to_shift['Trans.Time'].str.contains('r"\b\d{2} [A-Za-z]{3} \d{4}\b"', regex=True).notna()
shift_criteria.index

for index in shift_criteria.index:
    index = int(index)
    df_to_shift.iloc[index,:] = df_to_shift.iloc[index,:].shift()
df_to_shift = df_to_shift

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\APIN-PC\AppData\Local\Temp\ipykernel_31068\2062310880.py:1: SyntaxWarning: invalid escape sequence '\d'
  shift_criteria = df_to_shift['Trans.Time'].str.contains('r"\b\d{2} [A-Za-z]{3} \d{4}\b"', regex=True).notna()


### Joining Both Transformed Dataframe together

In [83]:
final_df = pd.concat([df_stable,df_to_shift], axis=0)
final_df = final_df.drop(columns=['level_0', 'index', 'Trans.Time', 'NoneDrop']).reset_index()
final_df = final_df.drop(columns='index')
final_df

,Value Date,Description,Debit/Credit(#),Balance(#),Channel,Transaction Reference
0,12 Mar 2024,Spend & Save Withdrawal,"+2,394.69","2,394.69",E-Channel,240312014770779834
1,12 Mar 2024,OWealth Deposit(AutoSave),-294.69,0.00,E-Channel,240312145795962609
2,27 Mar 2024,Transfer from NWAGBO CHINEDUM OBIOMA,"+4,500.00","4,500.00",E-Channel,000004240327184135633860074199
3,12 Apr 2024,Transfer from EBENEZER AYOMIKUN FALODUN,"+10,000.00","10,000.00",E-Channel,000014240412184328237230179735
4,12 Apr 2024,OWealth Deposit(AutoSave),"-10,000.00",0.00,E-Channel,240412146767712953
...,...,...,...,...,...,...
739,30 Mar 2025,OWealth Interest Earned,+0.09,0.71,E-Channel,2503309971yubCWGb81ei38grs5BvL
740,31 Mar 2025,OWealth Interest Earned,+0.09,0.80,E-Channel,250331991NjIlD3hc3FB1cfDlCnNp1
741,01 Apr 2025,OWealth Interest Earned,+0.09,0.89,E-Channel,250401993FeW6pQKUwq8gwbX6pXZ39
742,02 Apr 2025,OWealth Interest Earned,+0.09,0.98,E-Channel,250402995dD0FEHSCceB5u4PBUOpUQ


In [57]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 744 entries, 0 to 743
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Value Date             744 non-null    object
 1   Description            744 non-null    object
 2   Debit/Credit(#)        744 non-null    object
 3   Balance(#)             744 non-null    object
 4   Channel                744 non-null    object
 5   Transaction Reference  744 non-null    object
dtypes: object(6)
memory usage: 35.0+ KB


### Assigning Correct Datatypes

In [84]:
final_df['Value Date'] = pd.to_datetime(final_df['Value Date'])
final_df['Balance(#)'] = final_df['Balance(#)'].replace(r',', '', regex=True)
final_df['Balance(#)'] = final_df['Balance(#)'].replace(r'--', '0', regex=True).astype(float)
final_df['Debit/Credit(#)'] = final_df['Debit/Credit(#)'].replace(r',', '', regex=True).astype(float)

final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 744 entries, 0 to 743
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Value Date             744 non-null    datetime64[ns]
 1   Description            744 non-null    object        
 2   Debit/Credit(#)        744 non-null    float64       
 3   Balance(#)             744 non-null    float64       
 4   Channel                744 non-null    object        
 5   Transaction Reference  744 non-null    object        
dtypes: datetime64[ns](1), float64(2), object(3)
memory usage: 35.0+ KB


### Creating new credit and debit columns 

In [86]:
final_df['Credit'] = round(final_df['Debit/Credit(#)'].apply(lambda x:x if x > 0 else 0), 2)
final_df['Debit'] = abs(round(final_df['Debit/Credit(#)'].apply(lambda x:x if x < 0 else 0), 2))
final_df.head(4)

,Value Date,Description,Debit/Credit(#),Balance(#),Channel,Transaction Reference,Credit,Debit
0,2024-03-12,Spend & Save Withdrawal,2394.69,2394.69,E-Channel,240312014770779834,2394.69,0.00
1,2024-03-12,OWealth Deposit(AutoSave),-294.69,0.00,E-Channel,240312145795962609,0.00,294.69
2,2024-03-27,Transfer from NWAGBO CHINEDUM OBIOMA,4500.00,4500.00,E-Channel,000004240327184135633860074199,4500.00,0.00
3,2024-04-12,Transfer from EBENEZER AYOMIKUN FALODUN,10000.00,10000.00,E-Channel,000014240412184328237230179735,10000.00,0.00


In [88]:
# Drop Debit/Credit Column
opay_clean = final_df.drop('Debit/Credit(#)', axis=1)

### Transform cleaned data to specified format.

In [95]:
def transform_row_alat(row):
    return {
        "type": "debit" if int(row["Debit"]) > 0 else "credit",
        "amount": row["Debit"] if int(row["Debit"]) > 0 else row["Credit"],
        "narration": row["Description"].replace(r"'", ""),
        "date": row["Value Date"],
        "balance": row["Balance(#)"] 
    }
transformed_opay = opay_clean.apply(transform_row_alat, axis=1).tolist()
transformed_opay_df = pd.DataFrame(transformed_opay)
transformed_opay_df.head(5)
transformed_opay_df.to_csv('Transformed_bs.csv')

### Convert to json

In [92]:
transformed_opay_df.to_json(orient='records')

'[{"type":"credit","amount":2394.69,"narration":"Spend & Save Withdrawal","date":1710201600000,"balance":2394.69},{"type":"debit","amount":294.69,"narration":"OWealth Deposit(AutoSave)","date":1710201600000,"balance":0.0},{"type":"credit","amount":4500.0,"narration":"Transfer from NWAGBO CHINEDUM OBIOMA","date":1711497600000,"balance":4500.0},{"type":"credit","amount":10000.0,"narration":"Transfer from EBENEZER AYOMIKUN FALODUN","date":1712880000000,"balance":10000.0},{"type":"debit","amount":10000.0,"narration":"OWealth Deposit(AutoSave)","date":1712880000000,"balance":0.0},{"type":"credit","amount":45000.0,"narration":"Transfer from AROWOSEGBE VICTOR IYANUOLUWA","date":1712966400000,"balance":45000.0},{"type":"debit","amount":45000.0,"narration":"OWealth Deposit(AutoSave)","date":1712966400000,"balance":0.0},{"type":"credit","amount":2225.44,"narration":"Spend & Save Withdrawal","date":1713398400000,"balance":2225.44},{"type":"debit","amount":800.0,"narration":"Transfer to R A Paradi

### Reviewing Profit/Loss Statement

In [91]:
transformed_opay_df.groupby('type')['amount'].sum()
credit = 1276721.75
debit = 1264022.21

diff = credit - debit
diff

12699.540000000037

Expected difference

In [94]:
credit - 1276757.21

-35.45999999996275

##### More work is to be done. 
Credit is correct. But Debit seems to be missing some values

`expected difference is = -35.46`